In [1]:
import pandas as pd

# El archivo debe tener los encabezados en la primera fila:
# Nombre | Ciudad | Estado
locations = pd.read_excel(
    'ArchivosExcel/Lugares.xlsx'
)

# Verificar las columnas disponibles
print("Columnas encontradas:", locations.columns.tolist())

if 'Nombre' not in locations.columns:
    raise ValueError("El archivo debe contener una columna llamada 'Nombre'.")

locations.head()


Columnas encontradas: ['País LATAM', 'Nombre', 'Ciudad', 'Estado / Provincia / Departamento']


,País LATAM,Nombre,Ciudad,Estado / Provincia / Departamento
0,MX,Centro Médico Nacional Siglo XXI,Ciudad de México,NaN
1,MX,H. UNIVERSITARIO DR. JOSE ELEUTERIO GONZALEZ,NaN,NaN
2,MX,H. DE ESPECIALIDADES NO. 25 MONTERREY,Monterrey,NaN
3,MX,INP - Instituto Nacional de Pediatría,Ciudad de México,NaN
4,MX,Hospital Juárez de México,Ciudad de México,NaN


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import quote_plus
import re
import time

# Iniciar una sesión nueva de Chrome.
options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 12)

In [3]:
def clean_value(value):
    if pd.isna(value):
        return ''
    return str(value).strip()


def build_search_query(row):
    parts = [
        clean_value(row.get('Nombre', '')),
        clean_value(row.get('Ciudad', '')),
        clean_value(row.get('Estado', ''))
    ]
    return ' '.join(part for part in parts if part)


def get_google_name(driver):
    selector = 'h1.DUwDvf, h1[class*="fontHeadlineLarge"], h1'

    try:
        element = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, selector))
        )
        value = element.text.strip()

        if value:
            return value

    except Exception:
        pass

    return 'Name not found'


def get_address_from_maps(driver):
    selector = (
        '[data-item-id="address"], '
        'button[data-item-id="address"], '
        'button[aria-label^="Address:"], '
        'button[aria-label^="Dirección:"]'
    )

    try:
        element = WebDriverWait(driver, 4).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, selector))
        )

        value = element.get_attribute('aria-label') or element.text

        if value:
            value = re.sub(
                r'^(Address|Dirección):\s*',
                '',
                value,
                flags=re.IGNORECASE
            ).strip()

            if value:
                return value

    except Exception:
        pass

    return 'Address not found'


def extract_postal_code(address):
    if not address or address == 'Address not found':
        return 'Postal code not found'

    match = re.search(r'(?<!\d)(\d{5})(?:-\d{4})?(?!\d)', address)

    if match:
        return match.group(1)

    return 'Postal code not found'


def valid_coordinates(latitude, longitude):
    try:
        latitude = float(latitude)
        longitude = float(longitude)
        return -90 <= latitude <= 90 and -180 <= longitude <= 180
    except (TypeError, ValueError):
        return False


def extract_exact_coordinates_from_url(url):
    if not url:
        return None, None

    match = re.search(
        r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)',
        url
    )

    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            return lat, lon

    return None, None


def extract_viewport_center(url):
    if not url:
        return None, None

    match = re.search(
        r'@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)',
        url
    )

    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            return lat, lon

    return None, None


def _normalize_page_source(source):
    if not source:
        return ''

    normalized = source

    for _ in range(2):
        normalized = normalized.replace('\\\\"', '"')
        normalized = normalized.replace('\\"', '"')

    replacements = {
        r'\u0022': '"',
        r'\u003d': '=',
        r'\u0026': '&',
        '&quot;': '"',
        '&#34;': '"'
    }

    for old, new in replacements.items():
        normalized = normalized.replace(old, new)

    return normalized


def extract_structured_coordinates_from_page(driver):
    try:
        source = _normalize_page_source(driver.page_source)
    except Exception:
        return []

    candidates = []

    patterns = [
        r'"latitude"\s*:\s*(-?\d+(?:\.\d+)?)\s*,\s*"longitude"\s*:\s*(-?\d+(?:\.\d+)?)',
        r'"latitude"\s*:\s*(-?\d+(?:\.\d+)?).{0,180}?"longitude"\s*:\s*(-?\d+(?:\.\d+)?)',
        r'"longitude"\s*:\s*(-?\d+(?:\.\d+)?).{0,180}?"latitude"\s*:\s*(-?\d+(?:\.\d+)?)'
    ]

    for i, pattern in enumerate(patterns):
        for match in re.finditer(pattern, source, flags=re.DOTALL):
            if i < 2:
                lat, lon = float(match.group(1)), float(match.group(2))
            else:
                lon, lat = float(match.group(1)), float(match.group(2))

            if valid_coordinates(lat, lon):
                candidates.append((lat, lon))

    for match in re.finditer(
        r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)',
        source
    ):
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            candidates.append((lat, lon))

    unique = []
    seen = set()
    for lat, lon in candidates:
        key = (round(lat, 7), round(lon, 7))
        if key not in seen:
            seen.add(key)
            unique.append((lat, lon))

    return unique


def choose_best_candidate(candidates, driver):
    if not candidates:
        return None, None

    if len(candidates) == 1:
        return candidates[0]

    center_lat, center_lon = extract_viewport_center(driver.current_url)

    if center_lat is None or center_lon is None:
        return candidates[0]

    def squared_distance(candidate):
        lat, lon = candidate
        return (lat - center_lat) ** 2 + (lon - center_lon) ** 2

    return min(candidates, key=squared_distance)


def get_coordinates_from_maps(driver, result_url=None):
    # 1. Primero intenta obtener coordenadas exactas desde la URL.
    for label, candidate_url in [
        ('RESULT_URL_EXACT', result_url),
        ('CURRENT_URL_EXACT', driver.current_url)
    ]:
        lat, lon = extract_exact_coordinates_from_url(candidate_url)
        if lat is not None:
            return lat, lon, label

    # 2. Si no están en la URL, analiza el HTML una sola vez.
    page_candidates = extract_structured_coordinates_from_page(driver)

    if page_candidates:
        lat, lon = choose_best_candidate(page_candidates, driver)
        return lat, lon, 'PAGE_SOURCE'

    # 3. Como último recurso, usa el centro visible del mapa.
    for candidate_url in [driver.current_url, result_url]:
        lat, lon = extract_viewport_center(candidate_url)
        if lat is not None:
            return lat, lon, 'VIEWPORT_FALLBACK_REVIEW'

    return None, None, 'NOT_FOUND'


def search_google_maps(row):
    query = build_search_query(row)

    if not query:
        return {
            'Google_Name': '',
            'Google_Address': '',
            'Postal_Code': '',
            'Latitude': None,
            'Longitude': None,
            'Coordinate_Source': 'NOT_FOUND',
            'Search_Query': '',
            'Search_Status': 'EMPTY INPUT'
        }

    print(f'Buscando: {query}')

    try:
        # Navega directamente a la ficha del lugar, sin pasar por la lista de resultados
        url = (
            'https://www.google.com/maps/place/'
            f'{quote_plus(query)}'
        )

        driver.get(url)

        WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.TAG_NAME, 'body'))
        )

        body_text = driver.find_element(By.TAG_NAME, 'body').text.lower()

        if (
            'unusual traffic' in body_text
            or 'tráfico inusual' in body_text
            or 'captcha' in body_text
        ):
            return {
                'Google_Name': '',
                'Google_Address': '',
                'Postal_Code': '',
                'Latitude': None,
                'Longitude': None,
                'Coordinate_Source': 'NOT_FOUND',
                'Search_Query': query,
                'Search_Status': 'MANUAL REVIEW - CAPTCHA'
            }

        address = get_address_from_maps(driver)
        google_name = get_google_name(driver)
        postal_code = extract_postal_code(address)

        latitude, longitude, coordinate_source = get_coordinates_from_maps(
            driver,
            result_url=None
        )

        if address == 'Address not found':
            status = 'REVIEW'
        elif latitude is None or longitude is None:
            status = 'ADDRESS FOUND - COORDINATES REVIEW'
        elif coordinate_source == 'VIEWPORT_FALLBACK_REVIEW':
            status = 'COORDINATES FROM MAP CENTER - REVIEW'
        elif postal_code == 'Postal code not found':
            status = 'ADDRESS FOUND - CP REVIEW'
        else:
            status = 'OK'

        print(f'  Nombre: {google_name}')
        print(f'  Dirección: {address}')
        print(f'  CP: {postal_code}')
        print(f'  Latitud: {latitude}')
        print(f'  Longitud: {longitude}')
        print(f'  Fuente coordenadas: {coordinate_source}')
        print(f'  Estado: {status}')

        time.sleep(1)

        return {
            'Google_Name': google_name,
            'Google_Address': address,
            'Postal_Code': postal_code,
            'Latitude': latitude,
            'Longitude': longitude,
            'Coordinate_Source': coordinate_source,
            'Search_Query': query,
            'Search_Status': status
        }

    except Exception as e:
        print(f'Error con {query}: {e}')

        return {
            'Google_Name': '',
            'Google_Address': 'Error',
            'Postal_Code': '',
            'Latitude': None,
            'Longitude': None,
            'Coordinate_Source': 'NOT_FOUND',
            'Search_Query': query,
            'Search_Status': 'ERROR'
        }

In [4]:
# Procesar todas las filas del Excel.
# Esta parte escala automáticamente al número de registros del archivo.
results = locations.apply(
    search_google_maps,
    axis=1
)

# Agregar los resultados al DataFrame original.
locations['Google_Name'] = results.apply(lambda x: x['Google_Name'])
locations['Google_Address'] = results.apply(lambda x: x['Google_Address'])
locations['Postal_Code'] = results.apply(lambda x: x['Postal_Code'])
locations['Latitude'] = results.apply(lambda x: x['Latitude'])
locations['Longitude'] = results.apply(lambda x: x['Longitude'])
locations['Coordinate_Source'] = results.apply(lambda x: x['Coordinate_Source'])
locations['Search_Query'] = results.apply(lambda x: x['Search_Query'])
locations['Search_Status'] = results.apply(lambda x: x['Search_Status'])


Buscando: Centro Médico Nacional Siglo XXI Ciudad de México
  Nombre: Name not found
  Dirección: Address not found
  CP: Postal code not found
  Latitud: 19.4248704
  Longitud: -99.172352
  Fuente coordenadas: VIEWPORT_FALLBACK_REVIEW
  Estado: REVIEW
Buscando: H. UNIVERSITARIO DR. JOSE ELEUTERIO GONZALEZ
  Nombre: Name not found
  Dirección: Address not found
  CP: Postal code not found
  Latitud: 19.4248704
  Longitud: -99.172352
  Fuente coordenadas: VIEWPORT_FALLBACK_REVIEW
  Estado: REVIEW
Buscando: H. DE ESPECIALIDADES NO. 25 MONTERREY Monterrey
  Nombre: Name not found
  Dirección: Address not found
  CP: Postal code not found
  Latitud: 19.4248704
  Longitud: -99.172352
  Fuente coordenadas: VIEWPORT_FALLBACK_REVIEW
  Estado: REVIEW
Buscando: INP - Instituto Nacional de Pediatría Ciudad de México
  Nombre: Name not found
  Dirección: Address not found
  CP: Postal code not found
  Latitud: 19.4248704
  Longitud: -99.172352
  Fuente coordenadas: VIEWPORT_FALLBACK_REVIEW
  Estad

KeyboardInterrupt: 

In [ ]:
# Revisar resultados antes de exportar
locations[
    [
        'Nombre',
        'Google_Name',
        'Google_Address',
        'Postal_Code',
        'Latitude',
        'Longitude',
        'Coordinate_Source',
        'Search_Status'
    ]
].head(20)


,Nombre,Google_Name,Google_Address,Postal_Code,Latitude,Longitude,Coordinate_Source,Search_Status
0,Centro Médico Nacional Siglo XXI,Centro Médico Nacional Siglo XXI,"Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 0672...",06720,19.408343,-99.153994,CURRENT_URL_EXACT,OK
1,H. UNIVERSITARIO DR. JOSE ELEUTERIO GONZALEZ,Hospital Universitario Dr. José Eleuterio Gonz...,"Av. Madero y Gonzalitos S/N, Mitras Centro, Mo...",Postal code not found,25.688786,-100.350098,VIEWPORT_FALLBACK_REVIEW,COORDINATES FROM MAP CENTER - REVIEW
2,H. DE ESPECIALIDADES NO. 25 MONTERREY,IMSS - Unidad Médica de Alta Especialidad No. 25,"Av Fidel Velázquez s/n, Mitras Nte., 64180 Mon...",64180,25.716115,-100.350478,VIEWPORT_FALLBACK_REVIEW,COORDINATES FROM MAP CENTER - REVIEW
3,INP - Instituto Nacional de Pediatría,Instituto Nacional de Pediatría,"Av. Insurgentes Sur 3700-Letra C, Insurgentes ...",04530,19.306596,-99.185323,PAGE_SOURCE,OK
4,Hospital Juárez de México,Hospital Juárez de México,"Av Instituto Politécnico Nacional 5160, Magdal...",07760,19.481807,-99.135570,PAGE_SOURCE,OK
5,H. INFANTIL DE MEXICO FEDERICO GOMEZ,"Hospital Infantil de México, Federico Gómez","C. Dr.Márquez 162, Doctores, Cuauhtémoc, 06720...",06720,19.411491,-99.153131,RESULT_URL_EXACT,OK
6,ISSSTEP,ISSSTEP URGENCIAS,"Av Emiliano Zapata 4732, San Baltazar Campeche...",72550,19.022836,-98.201960,PAGE_SOURCE,OK
7,ABC OBSERVATORIO,Centro Médico ABC Observatorio,"Calle Sur 132 116, Las Américas, Álvaro Obregó...",01120,19.400246,-99.203669,PAGE_SOURCE,OK
8,Hospital Ángeles Lomas,Hospital Angeles Lomas,"Hospital Angeles Lomas, Vialidad de la Barranc...",52763,19.394815,-99.281493,PAGE_SOURCE,OK
9,CLÍNICA GÓMEZ ALMAGUER,Clínica Gómez Almaguer,"Loma Grande 2717, Lomas de San Francisco, 6471...",64710,25.665456,-100.346851,RESULT_URL_EXACT,OK


In [ ]:
# Cerrar Chrome después de procesar todas las ubicaciones
driver.quit()

# Guardar el archivo enriquecido
locations.to_excel(
    'ArchivosExcel/DireccionesRevisadas.xlsx',
    index=False
)

print('Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx')


Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx
